# Cohere Transcribe WER evaluation on FLEURS

This notebook measures sample WER locally on Apple Silicon. It loads the official Cohere BF16 checkpoint once through `mlx-audio`, then evaluates a deterministic sample from the requested FLEURS language configuration.

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from mlx_audio.stt.utils import load_model

repository_root = Path.cwd().resolve()
while not (repository_root / "macos" / "wer-test" / "wer_utils.py").is_file():
    if repository_root.parent == repository_root:
        raise RuntimeError(
            "Start Jupyter from the cohere-voice repository or a child directory."
        )
    repository_root = repository_root.parent

sys.path.insert(0, str(repository_root / "macos" / "wer-test"))
from wer_utils import (
    calculate_wer,
    get_fleurs_config,
    normalize_transcript,
    validate_positive,
)

In [ ]:
# Editable evaluation settings
MODEL_ID = "CohereLabs/cohere-transcribe-03-2026"
DATASET_ID = "google/fleurs"
LANGUAGE = "it"
SPLIT = "test"
SAMPLE_SIZE = 100
SEED = 42
BATCH_SIZE = 1
MAX_TOKENS = 256

validate_positive(SAMPLE_SIZE, "Sample size")
validate_positive(BATCH_SIZE, "Batch size")
DATASET_CONFIG = get_fleurs_config(LANGUAGE)
OUTPUT_DIR = repository_root / "macos" / "wer-test" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ID} / {DATASET_CONFIG} / {SPLIT}")
print(
    f"Language: {LANGUAGE}; sample size: {SAMPLE_SIZE}; seed: {SEED}; batch size: {BATCH_SIZE}"
)

In [ ]:
# Streaming avoids downloading an entire FLEURS language split before sampling.
dataset_stream = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split=SPLIT,
    streaming=True,
).shuffle(seed=SEED, buffer_size=1_000)
samples = list(islice(dataset_stream, SAMPLE_SIZE))

if len(samples) != SAMPLE_SIZE:
    raise RuntimeError(f"Requested {SAMPLE_SIZE} samples but received {len(samples)}.")

print(f"Collected {len(samples)} deterministic samples.")

In [ ]:
def extract_waveform(sample: dict) -> tuple[np.ndarray, int]:
    """Return a decoded FLEURS waveform and source sample rate."""
    audio = sample["audio"]
    if isinstance(audio, dict):
        return np.asarray(audio["array"]), int(audio["sampling_rate"])

    decoded = audio.get_all_samples()
    waveform = decoded.data
    if hasattr(waveform, "numpy"):
        waveform = waveform.numpy()
    return np.asarray(waveform), int(decoded.sample_rate)


def transcribe_batch(batch: list[dict], model) -> list[dict]:
    """Transcribe a batch, recording individual errors if a batch fails."""
    audio_items = [extract_waveform(sample) for sample in batch]
    arrays = [waveform for waveform, _ in audio_items]
    sample_rates = [sample_rate for _, sample_rate in audio_items]
    try:
        hypotheses = model.transcribe(
            audio_arrays=arrays,
            sample_rates=sample_rates,
            language=LANGUAGE,
            batch_size=BATCH_SIZE,
            max_tokens=MAX_TOKENS,
        )
        return [{"hypothesis": text, "error": None} for text in hypotheses]
    except Exception as batch_error:
        outcomes = []
        for waveform, sample_rate in audio_items:
            try:
                output = model.generate(
                    waveform,
                    sample_rate=sample_rate,
                    language=LANGUAGE,
                    max_tokens=MAX_TOKENS,
                )
                outcomes.append({"hypothesis": output.text, "error": None})
            except Exception as sample_error:
                outcomes.append(
                    {
                        "hypothesis": None,
                        "error": f"{type(batch_error).__name__}: {batch_error}; fallback: {type(sample_error).__name__}: {sample_error}",
                    }
                )
        return outcomes

In [ ]:
# Load the MLX model once. This requires Apple Silicon with Metal available.
model = load_model(MODEL_ID)
rows = []

for start in range(0, len(samples), BATCH_SIZE):
    batch = samples[start : start + BATCH_SIZE]
    outcomes = transcribe_batch(batch, model)
    for offset, (sample, outcome) in enumerate(zip(batch, outcomes)):
        reference = sample["transcription"]
        hypothesis = outcome["hypothesis"]
        normalized_reference = normalize_transcript(reference)
        normalized_hypothesis = normalize_transcript(hypothesis) if hypothesis else None
        sample_wer = (
            calculate_wer([normalized_reference], [normalized_hypothesis])
            if normalized_hypothesis is not None
            else None
        )
        rows.append(
            {
                "sample_index": start + offset,
                "dataset_id": sample["id"],
                "reference": reference,
                "hypothesis": hypothesis,
                "normalized_reference": normalized_reference,
                "normalized_hypothesis": normalized_hypothesis,
                "sample_wer": sample_wer,
                "error": outcome["error"],
            }
        )

    print(f"Completed {min(start + BATCH_SIZE, len(samples))}/{len(samples)} samples.")

In [ ]:
results = pd.DataFrame(rows)
successful = results[results["error"].isna()].copy()
if successful.empty:
    raise RuntimeError("No successful transcripts are available for WER calculation.")

corpus_wer = calculate_wer(
    successful["normalized_reference"].tolist(),
    successful["normalized_hypothesis"].tolist(),
)
summary = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "split": SPLIT,
    "language": LANGUAGE,
    "requested_samples": SAMPLE_SIZE,
    "successful_samples": len(successful),
    "failed_samples": int(results["error"].notna().sum()),
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "normalization": "NFKC, lowercase, punctuation removal, whitespace collapse",
    "corpus_wer": corpus_wer,
}

run_name = datetime.now(timezone.utc).strftime("wer_%Y%m%dT%H%M%SZ")
results_path = OUTPUT_DIR / f"{run_name}_details.csv"
summary_path = OUTPUT_DIR / f"{run_name}_summary.json"
results.to_csv(results_path, index=False)
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(pd.DataFrame([summary]))
display(
    results[["sample_index", "reference", "hypothesis", "sample_wer", "error"]].head(10)
)
print(f"Detailed results: {results_path}")
print(f"Summary: {summary_path}")